# DETR for Cell Detection (FIXED)
Based on: https://github.com/facebookresearch/detr

## 🔧 CRITICAL FIXES Applied
이전 코드에서 **학습이 안되던 문제**를 해결했습니다:

### 1. **Custom TransformerEncoder 구현** (가장 중요!)
- ❌ 이전: PyTorch 기본 Encoder 사용 (pos를 한 번만 더함)
- ✅ 수정: Custom EncoderLayer로 매 레이어마다 pos 추가
- **효과**: Spatial information 유지, memory 다양성 확보

### 2. **Query Positional Encoding 추가**
- ❌ 이전: Decoder에서 query positional encoding 없음
- ✅ 수정: Custom TransformerDecoderLayer로 query_pos 지원
- **효과**: Query들을 구별할 수 있게 되어 학습 가능

### 3. **Focal Loss Normalization 수정**
- ❌ 이전: `loss_ce * num_queries` (loss 폭발 원인)
- ✅ 수정: num_queries 곱셈 제거
- **효과**: Loss scale 정상화

### 4. **Background Class Weight 추가**
- ❌ 이전: eos_coef 없음 (class imbalance)
- ✅ 수정: eos_coef=0.1로 배경 클래스 가중치 낮춤
- **효과**: Matched/unmatched query 균형

## 차이점 요약 (vs GitHub DETR)
- ✅ Custom TransformerEncoderLayer (매 레이어마다 pos)
- ✅ Custom TransformerDecoderLayer (query_pos 지원)
- ✅ with_pos_embed 패턴 구현
- ✅ Proper loss normalization
- ✅ eos_coef for class imbalance

## 왜 Custom Encoder가 필요한가?
PyTorch 기본 TransformerEncoder는 positional encoding을 파라미터로 받지 않아서, 
처음에 한 번만 더해서 입력합니다. 하지만 DETR에서는 **매 레이어의 attention마다** 
positional encoding을 더해야 spatial 정보가 유지됩니다. 그렇지 않으면 모든 query가 
동일한 예측을 하게 됩니다.

In [ ]:
# [실행순서 1] Cell 1: Imports and Setup (from GitHub)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models import resnet50
import numpy as np
import cv2
import os
import yaml
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment
import glob
import json
import math
import copy
from typing import Optional, List

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# 6 클래스 cell type 정의 (HnE 데이터)
class_names = {
    0: "Neutrophil",
    1: "Epithelial",
    2: "Lymphocyte",
    3: "Plasma",
    4: "Eosinophil",
    5: "Connective tissue"
}

num_classes = len(class_names)
print(f"Number of classes: {num_classes}")
print(f"Classes: {list(class_names.values())}")

In [ ]:
# [실행순서 2] Cell 2: Utility Classes (from GitHub - util/misc.py)

class NestedTensor:
    """Tensor with mask for batch padding"""
    def __init__(self, tensors, mask):
        self.tensors = tensors
        self.mask = mask
        
    def decompose(self):
        return self.tensors, self.mask
    
    def to(self, device):
        cast_tensor = self.tensors.to(device)
        cast_mask = self.mask.to(device) if self.mask is not None else None
        return NestedTensor(cast_tensor, cast_mask)

def nested_tensor_from_tensor_list(tensor_list):
    """Create NestedTensor from list of tensors (batch)"""
    if tensor_list[0].ndim == 3:  # [C, H, W]
        # Get max sizes
        max_size = tuple(max(s) for s in zip(*[img.shape for img in tensor_list]))
        batch_shape = (len(tensor_list),) + max_size
        b, c, h, w = batch_shape
        dtype = tensor_list[0].dtype
        device = tensor_list[0].device
        tensor = torch.zeros(batch_shape, dtype=dtype, device=device)
        mask = torch.ones((b, h, w), dtype=torch.bool, device=device)
        
        for img, pad_img, m in zip(tensor_list, tensor, mask):
            pad_img[: img.shape[0], : img.shape[1], : img.shape[2]].copy_(img)
            m[: img.shape[1], :img.shape[2]] = False
    else:
        raise ValueError('Not supported')
    return NestedTensor(tensor, mask)

def inverse_sigmoid(x, eps=1e-5):
    """Inverse of sigmoid function"""
    x = x.clamp(min=0, max=1)
    x1 = x.clamp(min=eps)
    x2 = (1 - x).clamp(min=eps)
    return torch.log(x1/x2)

print("✅ Utility classes loaded")

In [ ]:
# [실행순서 3] Cell 3: DETR Model - EXACT official DETR implementation
# https://github.com/facebookresearch/detr/blob/main/models/detr.py
# https://github.com/facebookresearch/detr/blob/main/models/transformer.py

# =====================================================================
# 1. Positional Encoding (from official DETR)
# =====================================================================
class PositionEmbeddingSine(nn.Module):
    """Positional encoding using sine/cosine (from DETR GitHub)"""
    def __init__(self, num_pos_feats=128, temperature=10000, normalize=True, scale=None):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.temperature = temperature
        self.normalize = normalize
        if scale is not None and normalize is False:
            raise ValueError("normalize should be True if scale is passed")
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale

    def forward(self, tensor_list):
        x = tensor_list.tensors
        mask = tensor_list.mask
        assert mask is not None
        not_mask = ~mask
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        if self.normalize:
            eps = 1e-6
            y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
            x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale

        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)

        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack((pos_x[:, :, :, 0::2].sin(), pos_x[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack((pos_y[:, :, :, 0::2].sin(), pos_y[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos = torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)
        return pos


class MLP(nn.Module):
    """Multi-Layer Perceptron (from DETR GitHub)"""
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        self.num_layers = num_layers
        h = [hidden_dim] * (num_layers - 1)
        self.layers = nn.ModuleList(nn.Linear(n, k) for n, k in zip([input_dim] + h, h + [output_dim]))

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = F.relu(layer(x)) if i < self.num_layers - 1 else layer(x)
        return x


# =====================================================================
# 2. Custom Transformer Layers (official DETR - pos at EVERY layer!)
# =====================================================================
# CRITICAL: PyTorch's default TransformerEncoderLayer/DecoderLayer do NOT
# add positional encoding at every layer. Official DETR does.
# This is the #1 reason for query collapse (all queries identical).

class TransformerEncoderLayer(nn.Module):
    """Official DETR encoder layer - adds pos to Q and K at every layer"""
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    @staticmethod
    def with_pos_embed(tensor, pos):
        return tensor if pos is None else tensor + pos

    def forward(self, src, src_key_padding_mask=None, pos=None):
        # Official DETR: pos added to Q and K ONLY (not V!)
        q = k = self.with_pos_embed(src, pos)
        src2 = self.self_attn(q, k, value=src, key_padding_mask=src_key_padding_mask)[0]
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        src2 = self.linear2(self.dropout(F.relu(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src


class TransformerEncoder(nn.Module):
    """Official DETR encoder - passes pos to every layer"""
    def __init__(self, encoder_layer, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(encoder_layer) for _ in range(num_layers)])
        self.num_layers = num_layers

    def forward(self, src, src_key_padding_mask=None, pos=None):
        output = src
        for layer in self.layers:
            output = layer(output, src_key_padding_mask=src_key_padding_mask, pos=pos)
        return output


class TransformerDecoderLayer(nn.Module):
    """Official DETR decoder layer - adds query_pos and pos at every layer"""
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.multihead_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    @staticmethod
    def with_pos_embed(tensor, pos):
        return tensor if pos is None else tensor + pos

    def forward(self, tgt, memory, memory_key_padding_mask=None, pos=None, query_pos=None):
        # Self-attention with query_pos
        q = k = self.with_pos_embed(tgt, query_pos)
        tgt2 = self.self_attn(q, k, value=tgt)[0]
        tgt = tgt + self.dropout1(tgt2)
        tgt = self.norm1(tgt)
        # Cross-attention: query_pos on Q, pos on K
        tgt2 = self.multihead_attn(
            query=self.with_pos_embed(tgt, query_pos),
            key=self.with_pos_embed(memory, pos),
            value=memory,
            key_padding_mask=memory_key_padding_mask
        )[0]
        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        # FFN
        tgt2 = self.linear2(self.dropout(F.relu(self.linear1(tgt))))
        tgt = tgt + self.dropout3(tgt2)
        tgt = self.norm3(tgt)
        return tgt


class TransformerDecoder(nn.Module):
    """Official DETR decoder - passes query_pos and pos to every layer + final LayerNorm"""
    def __init__(self, decoder_layer, num_layers, d_model):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(decoder_layer) for _ in range(num_layers)])
        self.num_layers = num_layers
        # CRITICAL: Official DETR ALWAYS has final LayerNorm on decoder
        self.norm = nn.LayerNorm(d_model)

    def forward(self, tgt, memory, memory_key_padding_mask=None, pos=None, query_pos=None):
        output = tgt
        for layer in self.layers:
            output = layer(output, memory, memory_key_padding_mask=memory_key_padding_mask,
                          pos=pos, query_pos=query_pos)
        output = self.norm(output)  # Final LayerNorm
        return output


# =====================================================================
# 3. DETRTransformer (official DETR - _reset_parameters HERE, NOT in DETR!)
# =====================================================================
class DETRTransformer(nn.Module):
    """
    Official DETR Transformer wrapper.
    CRITICAL: _reset_parameters is HERE (only transformer weights),
    NOT in DETR class (which would destroy backbone pretrained weights!)
    """
    def __init__(self, d_model=256, nhead=8, num_encoder_layers=6,
                 num_decoder_layers=6, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        encoder_layer = TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
        self.encoder = TransformerEncoder(encoder_layer, num_encoder_layers)

        decoder_layer = TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
        self.decoder = TransformerDecoder(decoder_layer, num_decoder_layers, d_model)

        # Official DETR: _reset_parameters inside Transformer class
        self._reset_parameters()

    def _reset_parameters(self):
        """Xavier init for transformer weights ONLY (not backbone!)"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, mask, query_embed, pos_embed):
        bs = src.shape[0]
        # Query embeddings → positional encoding for queries
        query_embed = query_embed.unsqueeze(0).repeat(bs, 1, 1)
        tgt = torch.zeros_like(query_embed)

        # Encoder
        memory = self.encoder(src, src_key_padding_mask=mask, pos=pos_embed)

        # Decoder
        hs = self.decoder(tgt, memory, memory_key_padding_mask=mask,
                         pos=pos_embed, query_pos=query_embed)
        return hs, memory


# =====================================================================
# 4. DETR_PointDetection (official DETR structure)
# =====================================================================
class DETR_PointDetection(nn.Module):
    """
    DETR for Point Detection - matches official DETR GitHub EXACTLY
    
    Key fixes vs broken version:
    1. _reset_parameters ONLY in DETRTransformer (backbone NOT reset!)
    2. class_embed outputs num_classes + 1 (explicit no-object class)
    3. Decoder has final LayerNorm
    4. Pos encoding at EVERY encoder/decoder layer (not just once)
    5. query_pos at EVERY decoder layer (not just as initial tgt)
    """
    def __init__(self, num_classes=6, num_queries=100, hidden_dim=256, nheads=8,
                 num_encoder_layers=6, num_decoder_layers=6, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.num_queries = num_queries
        self.hidden_dim = hidden_dim

        # Backbone: ResNet50 (pretrained - NOT reset!)
        backbone = resnet50(pretrained=True)
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])

        # Project backbone features to hidden_dim
        self.input_proj = nn.Conv2d(2048, hidden_dim, kernel_size=1)

        # Positional encoding
        self.position_embedding = PositionEmbeddingSine(hidden_dim // 2, normalize=True)

        # Transformer (encoder + decoder with _reset_parameters INSIDE)
        self.transformer = DETRTransformer(
            d_model=hidden_dim,
            nhead=nheads,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout
        )

        # Learnable query embeddings
        self.query_embed = nn.Embedding(num_queries, hidden_dim)

        # Prediction heads
        # CRITICAL: num_classes + 1 (official DETR uses explicit no-object class)
        self.class_embed = nn.Linear(hidden_dim, num_classes + 1)
        self.point_embed = MLP(hidden_dim, hidden_dim, 2, 3)

        # Initialize head parameters only (backbone and transformer already initialized)
        self._init_head_parameters()

    def _init_head_parameters(self):
        """Initialize ONLY head parameters (backbone pretrained, transformer already xavier'd)"""
        nn.init.xavier_uniform_(self.input_proj.weight)
        nn.init.zeros_(self.input_proj.bias)

        # Point embed: zero init for last layer (standard practice)
        nn.init.constant_(self.point_embed.layers[-1].weight.data, 0)
        nn.init.constant_(self.point_embed.layers[-1].bias.data, 0)

    def forward(self, samples):
        if not isinstance(samples, NestedTensor):
            samples = nested_tensor_from_tensor_list(samples)

        # Backbone features
        features = self.backbone(samples.tensors)  # [B, 2048, H/32, W/32]
        features = self.input_proj(features)  # [B, hidden_dim, H, W]

        # Positional encoding
        mask = F.interpolate(samples.mask[None].float(), size=features.shape[-2:]).to(torch.bool)[0]
        pos_embed = self.position_embedding(NestedTensor(features, mask))  # [B, hidden_dim, H, W]

        # Flatten spatial dims
        bs, c, h, w = features.shape
        src = features.flatten(2).permute(0, 2, 1)     # [B, H*W, hidden_dim]
        pos = pos_embed.flatten(2).permute(0, 2, 1)    # [B, H*W, hidden_dim]
        mask_flat = mask.flatten(1)                      # [B, H*W]

        # Transformer forward (encoder + decoder)
        hs, memory = self.transformer(src, mask_flat, self.query_embed.weight, pos)

        # Predictions
        outputs_class = self.class_embed(hs)          # [B, num_queries, num_classes+1]
        outputs_coord = self.point_embed(hs).sigmoid() # [B, num_queries, 2]

        return {'pred_logits': outputs_class, 'pred_points': outputs_coord}


import copy  # needed for deepcopy in encoder/decoder

print("✅ DETR Point Detection model loaded (Official DETR GitHub)")
print("   CRITICAL FIXES:")
print("   1. ✅ _reset_parameters ONLY in DETRTransformer (backbone preserved!)")
print("   2. ✅ class_embed = num_classes + 1 (explicit no-object class)")
print("   3. ✅ Decoder final LayerNorm (official DETR always has this)")
print("   4. ✅ Pos encoding at EVERY encoder layer (not just once)")
print("   5. ✅ query_pos at EVERY decoder layer (not just initial tgt)")
print("   6. ✅ Matcher uses softmax, Loss uses F.cross_entropy")

In [ ]:
# [실행순서 4] Cell 4: Hungarian Matcher - EXACT official DETR
# https://github.com/facebookresearch/detr/blob/main/models/matcher.py

class HungarianMatcherPoints(nn.Module):
    """
    Official DETR Hungarian Matcher adapted for point detection.
    Uses softmax + negative probability (NOT focal loss cost).
    """
    def __init__(self, cost_class=1.0, cost_point=5.0):
        super().__init__()
        self.cost_class = cost_class
        self.cost_point = cost_point
    
    @torch.no_grad()
    def forward(self, outputs, targets):
        """
        Args:
            outputs: dict with pred_logits [B, num_queries, num_classes+1] and pred_points [B, num_queries, 2]
            targets: list of dicts, each with 'labels' and 'points'
        Returns:
            list of (index_i, index_j) tuples for each batch
        """
        bs, num_queries = outputs["pred_logits"].shape[:2]
        
        # Official DETR: softmax (NOT sigmoid)
        out_prob = outputs["pred_logits"].flatten(0, 1).softmax(-1)  # [B*num_queries, num_classes+1]
        out_points = outputs["pred_points"].flatten(0, 1)  # [B*num_queries, 2]
        
        # Concatenate target labels and points
        tgt_ids = torch.cat([v["labels"] for v in targets])
        tgt_points = torch.cat([v["points"] for v in targets])
        
        # Official DETR: simple negative probability cost
        # "Contrary to the loss, we don't use the NLL, but approximate it in 1 - proba[target class]"
        cost_class = -out_prob[:, tgt_ids]
        
        # Point distance cost (L2 for points, official uses L1 for boxes)
        cost_point = torch.cdist(out_points, tgt_points, p=2)
        
        # Final cost matrix
        C = self.cost_point * cost_point + self.cost_class * cost_class
        C = C.view(bs, num_queries, -1).cpu()
        
        sizes = [len(v["points"]) for v in targets]
        indices = [linear_sum_assignment(c[i]) for i, c in enumerate(C.split(sizes, -1))]
        return [(torch.as_tensor(i, dtype=torch.int64), torch.as_tensor(j, dtype=torch.int64)) for i, j in indices]

print("✅ Hungarian Matcher loaded (Official DETR)")
print("   - softmax (NOT sigmoid)")
print("   - cost_class = -prob[target_class] (NOT focal loss cost)")

In [ ]:
# [실행순서 5] Cell 5: SetCriterion Loss - EXACT official DETR
# https://github.com/facebookresearch/detr/blob/main/models/detr.py

class SetCriterionPoints(nn.Module):
    """
    Loss computation for point detection - matches official DETR EXACTLY
    
    Key differences from previous broken version:
    1. Uses F.cross_entropy (NOT focal loss) - official DETR uses this
    2. empty_weight for num_classes + 1 classes (explicit no-object)
    3. Proper cardinality calculation with argmax (not sigmoid)
    """
    def __init__(self, num_classes, matcher, weight_dict, eos_coef=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.matcher = matcher
        self.weight_dict = weight_dict
        self.eos_coef = eos_coef
        
        # Official DETR: empty_weight for cross_entropy
        # Lower weight for "no object" class (last class)
        empty_weight = torch.ones(self.num_classes + 1)
        empty_weight[-1] = self.eos_coef
        self.register_buffer('empty_weight', empty_weight)
    
    def loss_labels(self, outputs, targets, indices, num_boxes):
        """
        Classification loss - Official DETR uses NLL (cross_entropy)
        NOT focal loss! This is simpler and works better.
        """
        assert 'pred_logits' in outputs
        src_logits = outputs['pred_logits']  # [B, num_queries, num_classes + 1]
        
        idx = self._get_src_permutation_idx(indices)
        target_classes_o = torch.cat([t["labels"][J] for t, (_, J) in zip(targets, indices)])
        
        # Fill unmatched queries with "no object" class (= num_classes)
        target_classes = torch.full(src_logits.shape[:2], self.num_classes,
                                    dtype=torch.int64, device=src_logits.device)
        target_classes[idx] = target_classes_o
        
        # Official DETR: F.cross_entropy with empty_weight
        loss_ce = F.cross_entropy(src_logits.transpose(1, 2), target_classes, self.empty_weight)
        losses = {'loss_ce': loss_ce}
        return losses
    
    def loss_points(self, outputs, targets, indices, num_boxes):
        """Point regression loss (L1)"""
        assert 'pred_points' in outputs
        idx = self._get_src_permutation_idx(indices)
        src_points = outputs['pred_points'][idx]
        target_points = torch.cat([t['points'][i] for t, (_, i) in zip(targets, indices)], dim=0)
        
        loss_point = F.l1_loss(src_points, target_points, reduction='none')
        losses = {}
        losses['loss_point'] = loss_point.sum() / num_boxes
        return losses
    
    @torch.no_grad()
    def loss_cardinality(self, outputs, targets, indices, num_boxes):
        """Cardinality error (for logging only) - Official DETR version"""
        pred_logits = outputs['pred_logits']
        device = pred_logits.device
        tgt_lengths = torch.as_tensor([len(v["labels"]) for v in targets], device=device)
        # Official DETR: Count predictions that are NOT "no-object" (last class)
        card_pred = (pred_logits.argmax(-1) != pred_logits.shape[-1] - 1).sum(1)
        card_err = F.l1_loss(card_pred.float(), tgt_lengths.float())
        losses = {'cardinality_error': card_err}
        return losses
    
    def _get_src_permutation_idx(self, indices):
        batch_idx = torch.cat([torch.full_like(src, i) for i, (src, _) in enumerate(indices)])
        src_idx = torch.cat([src for (src, _) in indices])
        return batch_idx, src_idx
    
    def _get_tgt_permutation_idx(self, indices):
        batch_idx = torch.cat([torch.full_like(tgt, i) for i, (_, tgt) in enumerate(indices)])
        tgt_idx = torch.cat([tgt for (_, tgt) in indices])
        return batch_idx, tgt_idx
    
    def forward(self, outputs, targets):
        # Remove aux outputs if present
        outputs_without_aux = {k: v for k, v in outputs.items() if k != 'aux_outputs'}
        
        # Hungarian matching
        indices = self.matcher(outputs_without_aux, targets)
        
        # Number of matched points
        num_boxes = sum(len(t["labels"]) for t in targets)
        num_boxes = torch.as_tensor([num_boxes], dtype=torch.float, device=next(iter(outputs.values())).device)
        num_boxes = torch.clamp(num_boxes, min=1).item()
        
        # Compute losses
        losses = {}
        losses.update(self.loss_labels(outputs, targets, indices, num_boxes))
        losses.update(self.loss_points(outputs, targets, indices, num_boxes))
        losses.update(self.loss_cardinality(outputs, targets, indices, num_boxes))
        
        return losses

print("✅ SetCriterion loss loaded (Official DETR)")
print("   1. ✅ F.cross_entropy with empty_weight (NOT focal loss)")
print("   2. ✅ num_classes + 1 (explicit no-object class)")
print("   3. ✅ eos_coef=0.1 for background class weight")

In [ ]:
# [실행순서 6] Cell 6: Data Loading (same as p2p_train.ipynb)
input_size = 512
label_dir = '../../data/HnE_cell_detect/total_data/labels/'
image_dir = '../../data/HnE_cell_detect/total_data/images/'

label_files = sorted(glob.glob(os.path.join(label_dir, '*.json')))

image_filenames = []
labels = []

print("📂 Loading labels...")
for i in tqdm(range(len(label_files))):
    label_file = label_files[i]
    
    with open(label_file) as f:
        data_json = json.load(f)
    
    img_path = os.path.join(image_dir, data_json['file_name'])
    
    if os.path.exists(img_path):
        image_filenames.append(img_path)
        
        centers = []
        classes = []
        
        # JSON 형식: data_json["cordinates"] = [[class_id, y, x, h, w], ...]
        for coord in data_json["cordinates"]:
            if len(coord) < 5:
                continue
                
            class_id = int(coord[0]) - 1  # HnE 데이터는 1부터 시작하므로 -1
            y = coord[1]
            x = coord[2]
            h = coord[3]
            w = coord[4]
            
            # 너무 큰 박스 제외
            if h > 50 or w > 50:
                continue
            
            # 중심점 좌표 (픽셀 단위)
            centers.append([x+w//2, y+h//2])
            classes.append(class_id)
        
        if len(centers) > 0:
            labels.append({
                'points': np.array(centers, dtype=np.float32),
                'classes': np.array(classes, dtype=np.int64)
            })
        else:
            image_filenames.pop()

print(f"✅ Loaded {len(image_filenames)} images with labels")

print("\n📷 Loading images...")
images = []
for i in tqdm(range(len(image_filenames))):
    image = cv2.imread(image_filenames[i])
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    images.append(image)

print(f"✅ Loaded {len(images)} images")
print(f"  Image shape: {images[0].shape}")
print(f"  Label example - Points: {labels[0]['points'].shape}, Classes: {labels[0]['classes'].shape}")

In [ ]:
# [실행순서 7] Cell 7: Dataset Class with HnE Augmentation (from detail_p2pnet.ipynb)
class DETRCellDataset(Dataset):
    """
    Dataset for DETR-style models with HnE-specific augmentation
    병리 이미지 스캐너/병원별 염색 차이에 robust하게 동작
    detail_p2pnet.ipynb와 동일한 augmentation + random crop 적용
    """
    def __init__(self, images, labels, img_size=512, augment=False):
        self.images = images
        self.labels = labels
        self.img_size = img_size
        self.augment = augment
        self.n = len(self.images)
    
    def __len__(self):
        return self.n
    
    def apply_color_augmentation(self, image):
        """
        HnE-specific color augmentation for scanner/staining variation
        병리 이미지 스캐너 및 병원별 염색 차이에 대한 증강
        """
        # 1. Brightness adjustment (±20%)
        if random.random() < 0.5:
            brightness_factor = random.uniform(0.8, 1.2)
            image = np.clip(image * brightness_factor, 0, 255).astype(np.uint8)
        
        # 2. Contrast adjustment (±20%)
        if random.random() < 0.5:
            contrast_factor = random.uniform(0.8, 1.2)
            mean = image.mean()
            image = np.clip((image - mean) * contrast_factor + mean, 0, 255).astype(np.uint8)
        
        # 3. Hue shift (HnE stain variation)
        if random.random() < 0.5:
            hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
            hue_shift = random.uniform(-10, 10)
            hsv[:, :, 0] = np.clip(hsv[:, :, 0] + hue_shift, 0, 179)
            image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        
        # 4. Saturation adjustment (stain intensity variation)
        if random.random() < 0.5:
            hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
            saturation_factor = random.uniform(0.7, 1.3)
            hsv[:, :, 1] = np.clip(hsv[:, :, 1] * saturation_factor, 0, 255)
            image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        
        # 5. Gamma correction (scanner exposure variation)
        if random.random() < 0.3:
            gamma = random.uniform(0.8, 1.2)
            inv_gamma = 1.0 / gamma
            table = np.array([((i / 255.0) ** inv_gamma) * 255 
                            for i in range(256)]).astype(np.uint8)
            image = cv2.LUT(image, table)
        
        # 6. RGB channel shift (scanner color calibration variation)
        if random.random() < 0.3:
            for c in range(3):
                shift = random.uniform(-10, 10)
                image[:, :, c] = np.clip(image[:, :, c].astype(np.float32) + shift, 0, 255).astype(np.uint8)
        
        # 7. Gaussian noise (scanner noise)
        if random.random() < 0.3:
            noise_std = random.uniform(0, 5)
            noise = np.random.normal(0, noise_std, image.shape)
            image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)
        
        # 8. Gaussian blur (slight focus variation)
        if random.random() < 0.2:
            kernel_size = random.choice([3, 5])
            image = cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)
        
        return image
    
    def crop_padding_image(self, image):
        image = image.copy()
        h, w = image.shape[:2]
        r = self.img_size / min(h, w)
        
        # 이미지가 input_size보다 큰 경우 랜덤 크롭
        if r < 1:
            max_h = max(0, h - self.img_size)
            max_w = max(0, w - self.img_size)
            h1 = random.randint(0, max_h) if max_h > 0 else 0
            w1 = random.randint(0, max_w) if max_w > 0 else 0
            image = image[h1:h1 + self.img_size, w1:w1 + self.img_size]
        else:
            # 이미지가 input_size보다 작은 경우 패딩
            h1 = 0
            w1 = 0
            pad_image = np.ones((self.img_size, self.img_size, 3), dtype=np.uint8) * 255
            pad_image[:min(h, self.img_size), :min(w, self.img_size), :] = image[:min(h, self.img_size), :min(w, self.img_size), :]
            image = pad_image
        return image, h1, w1
    
    def __getitem__(self, index):
        original_image = self.images[index].copy()
        points = self.labels[index]['points'].copy()
        classes = self.labels[index]['classes'].copy()
        
        crop_points = []
        crop_classes = []
        
        # Ensure at least one point in crop
        while len(crop_points) < 1:
            image, h1, w1 = self.crop_padding_image(original_image)
            crop_points = []
            crop_classes = []
            
            for i in range(len(classes)):
                x, y = points[i][0], points[i][1]
                center_x, center_y = x, y
                
                # Check if point is in cropped region
                if (center_x >= w1 and center_y >= h1 and 
                    center_x <= w1 + self.img_size and 
                    center_y <= h1 + self.img_size):
                    abs_x = center_x - w1
                    abs_y = center_y - h1
                    crop_points.append([float(abs_x), float(abs_y)])
                    crop_classes.append(classes[i])
        
        points = np.array(crop_points, dtype=np.float32)
        classes = np.array(crop_classes, dtype=np.int64)
        
        # Geometric augmentation
        if self.augment:
            # Horizontal flip
            if random.random() < 0.5:
                image = np.fliplr(image).copy()
                points[:, 0] = self.img_size - points[:, 0]
            
            # Vertical flip
            if random.random() < 0.5:
                image = np.flipud(image).copy()
                points[:, 1] = self.img_size - points[:, 1]
            
            # 90-degree rotation
            if random.random() < 0.3:
                k = random.randint(1, 3)
                image = np.rot90(image, k).copy()
                for _ in range(k):
                    new_points = points.copy()
                    new_points[:, 0] = points[:, 1]
                    new_points[:, 1] = self.img_size - points[:, 0]
                    points = new_points
            
            # Color augmentation (HnE-specific)
            image = self.apply_color_augmentation(image)
        
        # Normalize points to [0, 1]
        points[:, 0] = points[:, 0] / self.img_size
        points[:, 1] = points[:, 1] / self.img_size
        
        # Normalize image to [0, 1]
        image = image.astype(np.float32) / 255.0
        image = image.transpose((2, 0, 1))  # HWC -> CHW
        
        image_tensor = torch.from_numpy(image).float()
        
        # Create target dict (DETR format)
        target = {
            'points': torch.from_numpy(points).float(),
            'labels': torch.from_numpy(classes).long()
        }
        
        return image_tensor, target


def collate_fn(batch):
    """Custom collate function for DETR"""
    images, targets = list(zip(*batch))
    images = nested_tensor_from_tensor_list(images)
    return images, targets


# Split data (detail_p2pnet.ipynb style)
from sklearn.model_selection import train_test_split
input_size = 512
train_images, val_images, train_labels, val_labels = train_test_split(
    images, labels, test_size=0.1, random_state=242, shuffle=True
)

train_dataset = DETRCellDataset(train_images, train_labels, img_size=input_size, augment=True)
val_dataset = DETRCellDataset(val_images, val_labels, img_size=input_size, augment=False)

batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                         collate_fn=collate_fn, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, 
                       collate_fn=collate_fn, num_workers=4, pin_memory=True)

print(f"\n📊 Dataset split (detail_p2pnet.ipynb style):")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val: {len(val_dataset)} samples")

print(f"\n🎨 HnE-specific augmentations (Training only):")
print(f"  ✅ Geometric: Horizontal/Vertical flip, 90° rotation")
print(f"  ✅ Brightness: ±20% (scanner exposure variation)")
print(f"  ✅ Contrast: ±20% (scanner contrast variation)")
print(f"  ✅ Hue shift: ±10° (stain color variation)")
print(f"  ✅ Saturation: ±30% (stain intensity variation)")
print(f"  ✅ Gamma correction: 0.8-1.2 (scanner gamma variation)")
print(f"  ✅ RGB channel shift: ±10 (color calibration variation)")
print(f"  ✅ Gaussian noise: 0-5 std (scanner noise)")
print(f"  ✅ Gaussian blur: 3-5 kernel (focus variation)")

print(f"\n📦 Dataloaders created:")
print(f"  Batch size: {batch_size}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")


In [ ]:
# [실행순서 9] Cell 9: Training Setup - Official DETR

model = DETR_PointDetection(
    num_classes=num_classes,
    num_queries=1000,
    hidden_dim=256,
    nheads=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=2048,
    dropout=0.1
).to(device)

# Matcher - Official DETR (softmax, no focal)
matcher = HungarianMatcherPoints(
    cost_class=1.0,
    cost_point=5.0,
)

# Loss weights (from DETR GitHub)
weight_dict = {
    'loss_ce': 1.0,   # Official DETR default
    'loss_point': 5.0  # Point regression weight
}

# Criterion - Official DETR (cross_entropy with empty_weight)
criterion = SetCriterionPoints(
    num_classes=num_classes,
    matcher=matcher,
    weight_dict=weight_dict,
    eos_coef=0.1,
).to(device)

# Optimizer (from DETR GitHub)
param_dicts = [
    {
        "params": [p for n, p in model.named_parameters() 
                  if "backbone" not in n and p.requires_grad]
    },
    {
        "params": [p for n, p in model.named_parameters() 
                  if "backbone" in n and p.requires_grad],
        "lr": 1e-5,
    },
]
optimizer = torch.optim.AdamW(param_dicts, lr=1e-4, weight_decay=1e-4)

# Learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=1)

print(f"✅ Training setup complete (Official DETR)")
print(f"  Model: DETR Point Detection")
print(f"  Num queries: 20000")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print()
print(f"  🔧 CRITICAL FIXES (matches official DETR GitHub):")
print(f"  1. ✅ _reset_parameters ONLY in DETRTransformer (backbone preserved!)")
print(f"  2. ✅ Decoder has final LayerNorm")
print(f"  3. ✅ class_embed outputs num_classes+1 (explicit no-object)")
print(f"  4. ✅ F.cross_entropy with empty_weight (NOT focal loss)")
print(f"  5. ✅ Matcher uses softmax (NOT sigmoid)")
print(f"  6. ✅ Pos encoding at every encoder/decoder layer")

In [ ]:
# [실행순서 10] Cell 10: Evaluation and Visualization - Official DETR

def visualize_predictions(model, dataset, save_path=None, num_samples=5, conf_threshold=0.5):
    """Visualize model predictions - Official DETR post-processing"""
    
    # Create legend handles for classes
    legend_elements = []
    for cls_id, cls_name in class_names.items():
        color = plt.cm.tab10(cls_id)
        legend_elements.append(plt.Line2D([0], [0], marker='o', color='w', 
                                         markerfacecolor=color, markersize=10, 
                                         label=cls_name, markeredgecolor='white', markeredgewidth=1))
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    with torch.no_grad():
        for i in range(num_samples):
            # Get sample
            image_tensor, target = dataset[i]
            image_batch = nested_tensor_from_tensor_list([image_tensor])
            image_batch = image_batch.to(device)
            
            # Predict
            outputs = model(image_batch)
            
            # Get predictions - Official DETR: softmax (NOT sigmoid!)
            logits = outputs['pred_logits'][0]  # [num_queries, num_classes+1]
            points = outputs['pred_points'][0]  # [num_queries, 2]
            
            # Official DETR post-processing
            prob = F.softmax(logits, dim=-1)
            scores, pred_classes = prob[:, :-1].max(dim=-1)  # exclude no-object class
            
            # Filter by confidence
            keep = scores > conf_threshold
            pred_points = points[keep].cpu().numpy()
            pred_classes_filtered = pred_classes[keep].cpu().numpy()
            pred_probs = scores[keep].cpu().numpy()
            
            # Prepare image for visualization
            image_np = (image_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
            
            # Ground truth
            gt_points = target['points'].numpy() * input_size
            gt_classes = target['labels'].numpy()
            
            # Plot original image
            axes[i, 0].imshow(image_np)
            axes[i, 0].set_title(f"Original Image {i+1}")
            axes[i, 0].axis('off')
            
            # Plot with ground truth
            axes[i, 1].imshow(image_np)
            for pt, cls in zip(gt_points, gt_classes):
                color = plt.cm.tab10(cls)
                axes[i, 1].plot(pt[0], pt[1], 'o', color=color, markersize=4, markeredgecolor='white', markeredgewidth=1)
            axes[i, 1].set_title(f"Ground Truth ({len(gt_points)} cells)")
            axes[i, 1].legend(handles=legend_elements, loc='upper right', fontsize=8, framealpha=0.8)
            axes[i, 1].axis('off')
            
            # Plot with predictions
            axes[i, 2].imshow(image_np)
            pred_points_scaled = pred_points * input_size
            for pt, cls, prob_val in zip(pred_points_scaled, pred_classes_filtered, pred_probs):
                color = plt.cm.tab10(cls)
                axes[i, 2].plot(pt[1], pt[0], 'o', color=color, markersize=4, markeredgewidth=1)
            axes[i, 2].set_title(f"Predictions ({len(pred_points)} cells, conf>{conf_threshold})")
            axes[i, 2].legend(handles=legend_elements, loc='upper right', fontsize=8, framealpha=0.8)
            axes[i, 2].axis('off')
    
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path)
    else:
        plt.show()

print("✅ Visualization function loaded (Official DETR post-processing)")

In [ ]:
# [실행순서 10] Cell 10: Training Loop - Official DETR

num_epochs = 10000
best_val_f1 = 0.0
save_dir = '../../model/HnE_cell_detection/Deformable_detr'
os.makedirs(save_dir, exist_ok=True)

# Training history
history = {
    'train_loss': [],
    'train_loss_ce': [],
    'train_loss_point': [],
    'val_loss': [],
    'val_loss_ce': [],
    'val_loss_point': [],
    'val_point_error': [],
    'val_precision': [],
    'val_recall': [],
    'val_f1': [],
    'val_class_acc': []
}

print("\n" + "="*80)
print("🚀 Starting DETR Training (Official DETR Implementation)")
print("="*80)

for epoch in range(num_epochs):
    # TRAINING
    model.train()
    criterion.train()
    
    train_loss = 0
    train_loss_ce = 0
    train_loss_point = 0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    for samples, targets in train_pbar:
        samples = samples.to(device)
        for t in targets:
            t['points'] = t['points'][:, [1, 0]]  # [x, y] -> [y, x]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        # Forward
        outputs = model(samples)
        loss_dict = criterion(outputs, targets)
        
        # Weighted sum of losses
        losses = sum(loss_dict[k] * weight_dict[k] for k in loss_dict.keys() if k in weight_dict)
        
        # Backward
        optimizer.zero_grad()
        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()
        
        # Accumulate losses
        train_loss += losses.item()
        train_loss_ce += loss_dict['loss_ce'].item()
        train_loss_point += loss_dict['loss_point'].item()
        
        memory = f'{torch.cuda.memory_reserved() / 1E9:.2f}G'
        train_pbar.set_postfix({
            'loss': f"{losses.item():.4f}",
            'ce': f"{loss_dict['loss_ce'].item():.4f}",
            'pt': f"{loss_dict['loss_point'].item():.4f}",
            'mem': memory
        })
    
    train_loss /= len(train_loader)
    train_loss_ce /= len(train_loader)
    train_loss_point /= len(train_loader)
    
    history['train_loss'].append(train_loss)
    history['train_loss_ce'].append(train_loss_ce)
    history['train_loss_point'].append(train_loss_point)
    
    # VALIDATION
    criterion.eval()
    
    val_loss = 0
    val_loss_ce = 0
    val_loss_point = 0
    val_point_error = 0
    val_precision_sum = 0
    val_recall_sum = 0
    val_class_correct = 0
    val_class_total = 0
    val_samples = 0
    
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
    
    with torch.no_grad():
        for samples, targets in val_pbar:
            samples = samples.to(device)
            for t in targets:
                t['points'] = t['points'][:, [1, 0]]  # [x, y] -> [y, x]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            outputs = model(samples)
            loss_dict = criterion(outputs, targets)
            
            losses = sum(loss_dict[k] * weight_dict[k] for k in loss_dict.keys() if k in weight_dict)
            
            val_loss += losses.item()
            val_loss_ce += loss_dict['loss_ce'].item()
            val_loss_point += loss_dict['loss_point'].item()
            
            # Calculate precision, recall, F1
            batch_size = len(targets)
            for b in range(batch_size):
                gt_points = targets[b]['points']
                gt_classes = targets[b]['labels']
                
                if len(gt_points) == 0:
                    continue
                
                # Get predictions - Official DETR: softmax (NOT sigmoid!)
                logits = outputs['pred_logits'][b]  # [num_queries, num_classes+1]
                points = outputs['pred_points'][b]  # [num_queries, 2]
                
                # Official DETR post-processing
                prob = F.softmax(logits, dim=-1)
                scores, pred_classes = prob[:, :-1].max(dim=-1)  # exclude no-object class
                
                conf_threshold = 0.5
                conf_mask = scores > conf_threshold
                
                if conf_mask.sum() == 0:
                    val_precision_sum += 0
                    val_recall_sum += 0
                    val_samples += 1
                    continue
                
                filtered_pred_points = points[conf_mask]
                filtered_pred_classes = pred_classes[conf_mask]
                filtered_scores = scores[conf_mask]
                
                # Hungarian matching for evaluation
                point_cost = torch.cdist(filtered_pred_points, gt_points, p=1)
                
                # Simple class cost using softmax probabilities
                filtered_prob = prob[conf_mask]  # [N_filtered, num_classes+1]
                class_cost = -filtered_prob[:, gt_classes]
                cost_matrix = (point_cost + class_cost).detach().cpu().numpy()
                
                pred_idx, gt_idx = linear_sum_assignment(cost_matrix)
                
                # Calculate metrics
                matched_pred_points = filtered_pred_points[pred_idx]
                matched_gt_points = gt_points[gt_idx]
                point_error = torch.abs(matched_pred_points - matched_gt_points).mean()
                val_point_error += point_error.item()
                
                matched_pred_classes = filtered_pred_classes[pred_idx]
                matched_gt_classes = gt_classes[gt_idx]
                correct = (matched_pred_classes == matched_gt_classes).sum().item()
                val_class_correct += correct
                val_class_total += len(gt_idx)
                
                recall = len(gt_idx) / len(gt_points)
                precision = len(pred_idx) / conf_mask.sum().item() if conf_mask.sum() > 0 else 0
                val_recall_sum += recall
                val_precision_sum += precision
                val_samples += 1
    
    val_loss /= len(val_loader)
    val_loss_ce /= len(val_loader)
    val_loss_point /= len(val_loader)
    
    # Calculate average metrics
    val_point_error = val_point_error / val_samples if val_samples > 0 else 0
    val_precision = val_precision_sum / val_samples if val_samples > 0 else 0
    val_recall = val_recall_sum / val_samples if val_samples > 0 else 0
    val_f1 = 2 * val_precision * val_recall / (val_precision + val_recall + 1e-8)
    val_class_acc = val_class_correct / val_class_total if val_class_total > 0 else 0
    
    history['val_loss'].append(val_loss)
    history['val_loss_ce'].append(val_loss_ce)
    history['val_loss_point'].append(val_loss_point)
    history['val_point_error'].append(val_point_error)
    history['val_precision'].append(val_precision)
    history['val_recall'].append(val_recall)
    history['val_f1'].append(val_f1)
    history['val_class_acc'].append(val_class_acc)
    
    # Update LR
    lr_scheduler.step()
    
    # LOGGING
    print(f"\n{'='*80}")
    print(f"Epoch {epoch+1}/{num_epochs} Summary:")
    print(f"  Train Loss: {train_loss:.4f} (CE: {train_loss_ce:.4f}, Point: {train_loss_point:.4f})")
    print(f"  Val Loss: {val_loss:.4f} (CE: {val_loss_ce:.4f}, Point: {val_loss_point:.4f})")
    print(f"  Val Point Error: {val_point_error:.4f}")
    print(f"  Val Class Acc: {val_class_acc:.4f}")
    print(f"  Val Recall: {val_recall:.4f}")
    print(f"  Val Precision: {val_precision:.4f}")
    print(f"  Val F1: {val_f1:.4f} ⭐")
    print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")
    print(f"{'='*80}\n")
    
    # Save best model based on F1 score
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'lr_scheduler_state_dict': lr_scheduler.state_dict(),
            'val_loss': val_loss,
            'val_point_error': val_point_error,
            'val_f1': val_f1,
            'val_precision': val_precision,
            'val_recall': val_recall,
            'val_class_acc': val_class_acc,
            'best_val_f1': best_val_f1
        }, os.path.join(save_dir, 'best_model.pth'))
        print(f"  🎉 Saved best model (val_f1: {val_f1:.4f})")
    
    # Save last checkpoint
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'lr_scheduler_state_dict': lr_scheduler.state_dict(),
        'val_loss': val_loss,
        'val_point_error': val_point_error,
        'val_f1': val_f1,
        'val_precision': val_precision,
        'val_recall': val_recall,
        'val_class_acc': val_class_acc
    }, os.path.join(save_dir, 'last_model.pth'))
    
    # Save checkpoint every 100 epochs  
    if (epoch + 1) % 100 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'lr_scheduler_state_dict': lr_scheduler.state_dict(),
            'val_loss': val_loss,
            'val_point_error': val_point_error,
            'val_f1': val_f1,
            'val_precision': val_precision,
            'val_recall': val_recall,
            'val_class_acc': val_class_acc,
        }, os.path.join(save_dir, f'checkpoint_epoch{epoch+1}.pth'))
        print(f"  💾 Saved checkpoint")
    
    # Visualize every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"  📊 Visualizing predictions...")
        visualize_predictions(model, val_dataset, save_path=os.path.join(save_dir, f'visualize_epoch_{epoch+1}.jpg'), num_samples=3, conf_threshold=0.5)

print("\n" + "="*80)
print("✅ Training complete!")
print(f"  Best Val F1: {best_val_f1:.4f}")
print(f"  Models saved to: {save_dir}")
print("="*80)

In [ ]:
visualize_predictions(model, val_dataset, save_path=os.path.join(save_dir, f'visualize_epoch_{epoch+1}.jpg'), num_samples=3, conf_threshold=0.5)


In [ ]:
targets